## Semiconductor Lab B7 - Task 3 & 4: Comparison with the Line Shape of Script B7

Same analysis as in `task3.ipynb`, with the line shape given in the script as the only change:

$$
I_\mathrm{sp}(\hbar\omega) \propto \omega^2 \sqrt{\hbar\omega - E_\mathrm{g}}\,\exp\left(-\frac{\hbar\omega - E_\mathrm{g}}{k_\mathrm{B}T}\right).
$$

Fit windows, logarithmic fit and $E_\mathrm{g} = E_\mathrm{peak} - k_\mathrm{B}T_\mathrm{c}/2$ are taken over
unchanged. The formula is applied to the measured signal as it is, so compared with $p_\lambda$ it
lacks the factor $hc/\lambda^2 \propto E^2$ of the change of variables from energy to wavelength.

In [ ]:
%matplotlib inline

from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from scipy import constants
from scipy.optimize import curve_fit

from b7.python.config import DATA_DIR, DOUBLE_COLUMN, PLOTS_DIR
from b7.python.helper import HC_EV_NM, read_spectrum

In [ ]:
data_dir: Path = DATA_DIR / "task_3_4_5"
export_dir: Path = PLOTS_DIR

### Import Data

In [ ]:
df_led: pd.DataFrame = read_spectrum(data_dir / "0.1mA.txt")
df_sub: pd.DataFrame = read_spectrum(data_dir / "9.9mA.txt")

### Fit Data

In [ ]:
WINDOW_LED: tuple[float, float] = (732.0, 775.0)
WINDOW_SUB: tuple[float, float] = (726.0, 768.0)

KB_EV: float = constants.k / constants.e  # Boltzmann constant in eV/K

In [ ]:
def log_script_line_shape(
    lambda_nm: npt.NDArray[np.float64], E_g: float, T_c: float, ln_A: float
) -> npt.NDArray[np.float64]:
    """Natural logarithm of the line shape of script B7, lambda_nm in nm, E_g in eV, T_c in K.

    The factor omega^2 is proportional to 1/lambda^2 and, as in `log_spontaneous_emission`,
    referred to 780 nm.
    """
    x: npt.NDArray[np.float64] = np.clip(HC_EV_NM / lambda_nm - E_g, 1e-12, None)
    return ln_A - 2 * np.log(lambda_nm / 780.0) + 0.5 * np.log(x) - x / (KB_EV * T_c)

In [ ]:
mask_led: pd.Series = df_led["lambda_nm"].between(*WINDOW_LED) & (df_led["signal"] > 0)
lambda_led: npt.NDArray[np.float64] = df_led.loc[mask_led, "lambda_nm"].to_numpy()
ln_p_led: npt.NDArray[np.float64] = np.log(df_led.loc[mask_led, "signal"].to_numpy())

lambda_peak_led: float = df_led.loc[df_led["signal"].idxmax(), "lambda_nm"]
E_peak_led: float = HC_EV_NM / lambda_peak_led

In [ ]:
# Same window, data and peak as the main fit of the LED spectrum
(T_c_led_b7, ln_A_led_b7), cov_led_b7 = curve_fit(
    lambda lambda_nm, T_c, ln_A: log_script_line_shape(lambda_nm, E_peak_led - KB_EV * T_c / 2, T_c, ln_A),
    lambda_led,
    ln_p_led,
    p0=[300.0, 0.0],
    bounds=([50.0, -60.0], [3000.0, 60.0]),
)
T_c_led_b7_err, _ = np.sqrt(np.diag(cov_led_b7))
E_g_led_b7: float = E_peak_led - KB_EV * T_c_led_b7 / 2
E_g_led_b7_err: float = KB_EV * T_c_led_b7_err / 2

fit_led_b7 = {
    "n_points": int(mask_led.sum()),
    "E_peak": E_peak_led,
    "E_g": E_g_led_b7,
    "E_g_err": E_g_led_b7_err,
    "T_c": T_c_led_b7,
    "T_c_err": T_c_led_b7_err,
    "kT_meV": KB_EV * T_c_led_b7 * 1e3,
}
pprint(fit_led_b7)

In [ ]:
mask_sub: pd.Series = df_sub["lambda_nm"].between(*WINDOW_SUB) & (df_sub["signal"] > 0)
lambda_sub: npt.NDArray[np.float64] = df_sub.loc[mask_sub, "lambda_nm"].to_numpy()
ln_p_sub: npt.NDArray[np.float64] = np.log(df_sub.loc[mask_sub, "signal"].to_numpy())

lambda_peak_sub: float = df_sub.loc[df_sub["signal"].idxmax(), "lambda_nm"]
E_peak_sub: float = HC_EV_NM / lambda_peak_sub

In [ ]:
# Same window, data and peak as the main fit of the 9.9 mA spectrum
(T_c_sub_b7, ln_A_sub_b7), cov_sub_b7 = curve_fit(
    lambda lambda_nm, T_c, ln_A: log_script_line_shape(lambda_nm, E_peak_sub - KB_EV * T_c / 2, T_c, ln_A),
    lambda_sub,
    ln_p_sub,
    p0=[300.0, 0.0],
    bounds=([50.0, -60.0], [3000.0, 60.0]),
)
T_c_sub_b7_err, _ = np.sqrt(np.diag(cov_sub_b7))
E_g_sub_b7: float = E_peak_sub - KB_EV * T_c_sub_b7 / 2
E_g_sub_b7_err: float = KB_EV * T_c_sub_b7_err / 2

fit_sub_b7 = {
    "n_points": int(mask_sub.sum()),
    "E_peak": E_peak_sub,
    "E_g": E_g_sub_b7,
    "E_g_err": E_g_sub_b7_err,
    "T_c": T_c_sub_b7,
    "T_c_err": T_c_sub_b7_err,
    "kT_meV": KB_EV * T_c_sub_b7 * 1e3,
}
pprint(fit_sub_b7)

### Plot Data

#### Preparation

In [ ]:
# Smooth line shape curve over each fitted window, padded a bit on either side
pad = 100.0
lambda_led_curve = np.linspace(lambda_led.min() - pad, lambda_led.max() + pad, 400)
lambda_sub_curve = np.linspace(lambda_sub.min() - pad, lambda_sub.max() + pad, 400)
p_led_b7_curve = np.exp(log_script_line_shape(lambda_led_curve, E_g_led_b7, T_c_led_b7, ln_A_led_b7))
p_sub_b7_curve = np.exp(log_script_line_shape(lambda_sub_curve, E_g_sub_b7, T_c_sub_b7, ln_A_sub_b7))

# 1/lambda^2 and square root divided out, so the carrier temperature is simply the slope of a
# straight line in E; only points above E_g with positive signal have a logarithm
above_led_b7: pd.Series = (df_led["E_eV"] > E_g_led_b7) & (df_led["signal"] > 0)
E_led_b7_flat = df_led.loc[above_led_b7, "E_eV"].to_numpy()
lambda_led_b7_flat = df_led.loc[above_led_b7, "lambda_nm"].to_numpy()
ln_p_led_b7_flat = (
    np.log(df_led.loc[above_led_b7, "signal"].to_numpy())
    + 2 * np.log(lambda_led_b7_flat / 780.0)
    - 0.5 * np.log(E_led_b7_flat - E_g_led_b7)
)
ln_p_led_b7_flat_fit = ln_A_led_b7 - (E_led_b7_flat - E_g_led_b7) / (KB_EV * T_c_led_b7)

above_sub_b7: pd.Series = (df_sub["E_eV"] > E_g_sub_b7) & (df_sub["signal"] > 0)
E_sub_b7_flat = df_sub.loc[above_sub_b7, "E_eV"].to_numpy()
lambda_sub_b7_flat = df_sub.loc[above_sub_b7, "lambda_nm"].to_numpy()
ln_p_sub_b7_flat = (
    np.log(df_sub.loc[above_sub_b7, "signal"].to_numpy())
    + 2 * np.log(lambda_sub_b7_flat / 780.0)
    - 0.5 * np.log(E_sub_b7_flat - E_g_sub_b7)
)
ln_p_sub_b7_flat_fit = ln_A_sub_b7 - (E_sub_b7_flat - E_g_sub_b7) / (KB_EV * T_c_sub_b7)


In [ ]:
with plt.rc_context(DOUBLE_COLUMN):
    fig: Figure = plt.figure(layout="constrained")
    axs: dict[str, Axes] = fig.subplot_mosaic([["led", "flat"], ["sub", "flat"]])
    ax_led: Axes = axs["led"]
    ax_sub: Axes = axs["sub"]
    ax_flat: Axes = axs["flat"]
    ax_led.sharex(ax_sub)

    ax_led.scatter(df_led["lambda_nm"], df_led["signal"], s=4, color="tab:blue", label=r"\qty{0.1}{\milli\ampere}")
    ax_led.plot(lambda_led_curve, p_led_b7_curve, color="navy", linestyle="--", zorder=-1)
    ax_led.set_yscale("log")
    ax_led.set_ylim(1e-4, 1e-1)
    ax_led.set_ylabel(r"$I$ [arb. u.]")
    ax_led.tick_params(labelbottom=False)
    ax_led.legend(loc="upper left")

    ax_sub.scatter(df_sub["lambda_nm"], df_sub["signal"], s=4, color="tab:red", label=r"\qty{9.9}{\milli\ampere}")
    ax_sub.plot(lambda_sub_curve, p_sub_b7_curve, color="darkred", linestyle="--", zorder=-1)
    ax_sub.set_yscale("log")
    ax_sub.set_ylim(1e-4, 5)
    ax_sub.set_ylabel(r"$I$ [arb. u.]")
    ax_sub.set_xlabel(r"$\lambda$ [\unit{\nano\meter}]")
    ax_sub.legend(loc="upper left")

    ax_flat.scatter(E_led_b7_flat, ln_p_led_b7_flat, s=6, color="tab:blue", label=r"\qty{0.1}{\milli\ampere}")
    ax_flat.scatter(E_sub_b7_flat, ln_p_sub_b7_flat, s=6, color="tab:red", label=r"\qty{9.9}{\milli\ampere}")
    ax_flat.plot(
        E_led_b7_flat,
        ln_p_led_b7_flat_fit,
        color="navy",
        linestyle="--",
        zorder=-1,
        label=rf"$T_\mathrm{{c}} = \qty{{{T_c_led_b7:.0f}}}{{\kelvin}}$",
    )
    ax_flat.plot(
        E_sub_b7_flat,
        ln_p_sub_b7_flat_fit,
        color="darkred",
        linestyle="--",
        zorder=-1,
        label=rf"$T_\mathrm{{c}} = \qty{{{T_c_sub_b7:.0f}}}{{\kelvin}}$",
    )
    ax_flat.set_ylim(-11, 3)
    ax_flat.set_xlabel(r"$E$ [\unit{\electronvolt}]")
    ax_flat.set_ylabel(r"$\ln I + 2 \ln \lambda - \frac{1}{2} \ln (E - E_\mathrm{g})$")
    ax_flat.legend(loc="upper right", fontsize=8)

    plt.show()